# 🍏 Retrieval-Augmented Generation (RAG) with AIProjectClient 🍎

In this notebook, we'll demonstrate a **basic RAG** flow using:
- **`azure-ai-projects`** (AIProjectClient)
- **`azure-ai-inference`** (Embeddings, ChatCompletions)
- **`azure-ai-search`** (for vector or hybrid search)

Our theme is **Health & Fitness** 🍏 so we’ll create a simple set of health tips, embed them, store them in a search index, then do a query that retrieves relevant tips, and pass them to an LLM to produce a final answer.

> **Disclaimer**: This is not medical advice. For real health questions, consult a professional.

## What is RAG?
Retrieval-Augmented Generation (RAG) is a technique where the LLM (Large Language Model) uses relevant retrieved text chunks from your data to craft a final answer. This helps ground the model's response in real data, reducing hallucinations.


<img src="./seq-diagrams/rag.png" width="30%"/>

## 1. Setup
We'll import libraries, load environment variables, and create an `AIProjectClient`.

In [28]:
import os
import json
from pathlib import Path

# Azure SDKs
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.core.credentials import AzureKeyCredential

# ------------------------------------------------------
# Utility: Find cred.json
# ------------------------------------------------------
def find_cred_json(start_path):
    current = Path(start_path)
    while current != current.parent:
        cred_file = current / "cred.json"
        if cred_file.exists():
            return str(cred_file)
        current = current.parent
    return None

# ------------------------------------------------------
# Load configuration
# ------------------------------------------------------
file_path = find_cred_json(os.getcwd())
if not file_path:
    raise FileNotFoundError("cred.json not found in current or parent directories")

with open(file_path, "r") as f:
    cfg = json.load(f)

# Required values
endpoint = cfg.get("PROJECT_ENDPOINT")   # Must be endpoint (not connection string)
model_deployment_name = cfg.get("MODEL_DEPLOYMENT_NAME", "gpt-4o")
embedding_model_name = cfg.get("EMBEDDING_MODEL_DEPLOYMENT_NAME", "text-embedding-3-small")

# Search settings
search_service_endpoint = cfg.get("SEARCH_ENDPOINT")       # e.g. "https://<your-service>.search.windows.net"
search_api_key = cfg.get("SEARCH_API_KEY")
search_index_name = cfg.get("SEARCH_INDEX_NAME", "psitronaiserach")

print("Project Endpoint:", endpoint)
print("Chat Model Deployment:", model_deployment_name)
print("Embedding Model Deployment:", embedding_model_name)
print("Search Index Name:", search_index_name)

# ------------------------------------------------------
# Initialize Project + OpenAI client
# ------------------------------------------------------
credential = DefaultAzureCredential()
project = AIProjectClient(endpoint=endpoint, credential=credential)

# OpenAI-compatible client
openai_client = project.get_openai_client(api_version="2024-10-21")

print("✅ AIProjectClient + OpenAI client initialized")

# ------------------------------------------------------
# Initialize Search Clients
# ------------------------------------------------------
#search_cred = AzureKeyCredential(search_api_key)
#search_client = SearchClient(endpoint=search_service_endpoint, index_name=search_index_name, credential=search_cred)
#index_client = SearchIndexClient(endpoint=search_service_endpoint, credential=search_cred)

print("✅ SearchClient and IndexClient initialized")

# ------------------------------------------------------
# Example helpers
# ------------------------------------------------------
def chat(prompt: str):
    response = openai_client.chat.completions.create(
        model=model_deployment_name,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

def embed_text(text: str):
    response = openai_client.embeddings.create(
        model=embedding_model_name,
        input=text
    )
    return response.data[0].embedding
# ------------------------------------------------------
# Run demo
# ------------------------------------------------------
if __name__ == "__main__":
    print("💬 Chat:", chat("Hello, how are you?"))
    print("📐 Embedding length:", len(embed_text("MLOps is awesome")))


Project Endpoint: https://sarath-8734-resource.services.ai.azure.com/api/projects/sarath-8734
Chat Model Deployment: gpt-4o
Embedding Model Deployment: text-embedding-3-small
Search Index Name: psitronaiserach
✅ AIProjectClient + OpenAI client initialized
✅ SearchClient and IndexClient initialized
💬 Chat: Hello! I'm just a program, so I don't have feelings, but thank you for asking. 😊 How can I assist you today?
📐 Embedding length: 1536


## 2. Create Sample Health Data
We'll create a few short doc chunks. In a real scenario, you might read from CSV or PDFs, chunk them up, embed them, and store them in your search index.


In [29]:
health_tips = [
    {
        "id": "doc1",
        "content": "Daily 30-minute walks help maintain a healthy weight and reduce stress.",
        "source": "General Fitness"
    },
    {
        "id": "doc2",
        "content": "Stay hydrated by drinking 8-10 cups of water per day.",
        "source": "General Fitness"
    },
    {
        "id": "doc3",
        "content": "Consistent sleep patterns (7-9 hours) improve muscle recovery.",
        "source": "General Fitness"
    },
    {
        "id": "doc4",
        "content": "For cardio endurance, try interval training like HIIT.",
        "source": "Workout Advice"
    },
    {
        "id": "doc5",
        "content": "Warm up with dynamic stretches before running to reduce injury risk.",
        "source": "Workout Advice"
    },
    {
        "id": "doc6",
        "content": "Balanced diets typically include protein, whole grains, fruits, vegetables, and healthy fats.",
        "source": "Nutrition"
    },
]
print("Created a small list of health tips.")

Created a small list of health tips.


## 3.0. Create or Reset the Index
When creating a vector field in Azure AI Search, the **field definition** must include a `vector_search_profile` property that points to a matching profile name in your vector search settings.

We'll define a helper function to create (or reset) a vector index with an HNSW algorithm config.


In [30]:
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    HnswParameters,
    VectorSearchAlgorithmKind,
    VectorSearchAlgorithmMetric,
    VectorSearchProfile,
)

def create_healthtips_index(
        endpoint: str, api_key: str, index_name: str, 
        dimension: int = 1536 # if using text-embedding-3-small
        ):
    """Create or update a search index for health tips with vector search capability."""
    
    index_client = SearchIndexClient(endpoint=endpoint, credential=AzureKeyCredential(api_key))
    
    # Try to delete existing index
    try:
        index_client.delete_index(index_name)
        print(f"Deleted existing index: {index_name}")
    except Exception:
        pass  # Index doesn't exist yet
        
    # Define vector search configuration
    vector_search = VectorSearch(
        algorithms=[
            HnswAlgorithmConfiguration(
                name="myHnsw",
                kind=VectorSearchAlgorithmKind.HNSW,
                parameters=HnswParameters(
                    m=4,
                    ef_construction=400,
                    ef_search=500,
                    metric=VectorSearchAlgorithmMetric.COSINE
                )
            )
        ],
        profiles=[
            VectorSearchProfile(
                name="myHnswProfile",
                algorithm_configuration_name="myHnsw"
            )
        ]
    )
    
    # Define fields
    fields = [
        SimpleField(name="id", type=SearchFieldDataType.String, key=True),
        SearchableField(name="content", type=SearchFieldDataType.String),
        SimpleField(name="source", type=SearchFieldDataType.String),
        SearchField(
            name="embedding", 
            type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
            vector_search_dimensions=dimension,
            vector_search_profile_name="myHnswProfile" 
        ),
    ]
    
    # Create index definition
    index_def = SearchIndex(
        name=index_name,
        fields=fields,
        vector_search=vector_search
    )
    
    # Create the index
    index_client.create_index(index_def)
    print(f"✅ Created or reset index: {index_name}")

## 3.1. Create Index & Upload Health Tips 🏋️

Now we'll put our health tips into action by:
1. **Creating a search connection** to Azure AI Search
2. **Building our index** with vector search capability
3. **Generating embeddings** for each health tip
4. **Uploading** the tips with their embeddings

This creates our knowledge base that we'll search through later. Think of it as building our 'fitness library' that our AI assistant can reference! 📚💪

In [31]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

project_client = AIProjectClient(
    endpoint=cfg["PROJECT_ENDPOINT"],
    credential=DefaultAzureCredential(),
)

In [32]:
print("List all connections:")
for connection in project_client.connections.list():
    print(connection)

List all connections:
{'name': 'psitrontechnologies', 'id': '/subscriptions/1c2fd79b-ad21-4ad0-8d53-12de16650452/resourceGroups/rg-sarath-8734/providers/Microsoft.CognitiveServices/accounts/sarath-8734-resource/projects/sarath-8734/connections/psitrontechnologies', 'type': 'CognitiveSearch', 'target': 'https://psitrontechnologies.search.windows.net/', 'isDefault': True, 'credentials': {'type': 'ApiKey'}, 'metadata': {'type': 'azure_ai_search', 'ApiType': 'Azure', 'ResourceId': '/subscriptions/1c2fd79b-ad21-4ad0-8d53-12de16650452/resourceGroups/rg-sarath-8734/providers/Microsoft.Search/searchServices/psitrontechnologies', 'ApiVersion': '2024-05-01-preview', 'DeploymentApiVersion': '2023-11-01'}}


In [33]:
from azure.ai.projects.models import ConnectionType
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential

# Step 1: Get search connection
search_conn = project_client.connections.get_default(
    connection_type=ConnectionType.AZURE_AI_SEARCH,
    include_credentials=True
)
if not search_conn:
    raise RuntimeError("❌ No default Azure AI Search connection found!")
print("✅ Got search connection")

# Step 2: Get OpenAI-compatible client instead of inference client
openai_client = project_client.get_openai_client(api_version="2024-10-21")
print("✅ Created OpenAI client")

# Example doc
sample_doc = health_tips[0]

# Create embedding
emb_response = openai_client.embeddings.create(
    model=embedding_model_name,
    input=sample_doc["content"]
)
embedding_length = len(emb_response.data[0].embedding)
print(f"✅ Got embedding length: {embedding_length}")

# Step 3: Create the index
create_healthtips_index(
    endpoint=search_service_endpoint,
    api_key=search_api_key,
    index_name=search_index_name,
    dimension=embedding_length   # for text-embedding-3-large or your model
)

# Step 4: Create search client for uploading documents
search_client = SearchClient(
    endpoint=search_service_endpoint,
    index_name=search_index_name,
    credential=AzureKeyCredential(search_api_key)
)
print("✅ Created search client")

# Step 5: Embed and upload documents
search_docs = []
for doc in health_tips:
    emb_response = openai_client.embeddings.create(
        model=embedding_model_name,
        input=doc["content"]
    )
    emb_vec = emb_response.data[0].embedding
    
    search_docs.append({
        "id": doc["id"],
        "content": doc["content"],
        "source": doc["source"],
        "embedding": emb_vec,
    })

# Upload docs
result = search_client.upload_documents(documents=search_docs)
print(f"✅ Uploaded {len(search_docs)} documents to search index '{search_index_name}'")


✅ Got search connection
✅ Created OpenAI client
✅ Got embedding length: 1536
Deleted existing index: psitronaiserach
✅ Created or reset index: psitronaiserach
✅ Created search client
✅ Uploaded 6 documents to search index 'psitronaiserach'


## 4. Basic RAG Flow
### 4.1. Retrieve
When a user queries, we:
1. Embed user question.
2. Search vector index with that embedding to get top docs.

### 4.2. Generate answer
We then pass the retrieved docs to the chat model.

> In a real scenario, you'd have a more advanced approach to chunking & summarizing. We'll keep it simple.


In [34]:
from azure.search.documents.models import VectorizedQuery

def rag_chat(query: str, top_k: int = 3) -> str:
    # 1) Embed user query
    emb_response = openai_client.embeddings.create(
        model=embedding_model_name,
        input=query
    )
    user_vec = emb_response.data[0].embedding

    # 2) Vector search using VectorizedQuery
    vector_query = VectorizedQuery(
        vector=user_vec,
        k_nearest_neighbors=top_k,
        fields="embedding"
    )

    results = search_client.search(
        search_text="",  # Optional hybrid text query
        vector_queries=[vector_query],
        select=["content", "source"]
    )

    # Gather top documents
    top_docs_content = []
    for r in results:
        c = r["content"]
        s = r["source"]
        top_docs_content.append(f"Source: {s} => {c}")

    # 3) Chat with retrieved docs
    system_text = (
        "You are a health & fitness assistant.\n"
        "Answer user questions using ONLY the text from these docs.\n"
        "Docs:\n"
        + "\n".join(top_docs_content)
        + "\nIf unsure, say 'I'm not sure'.\n"
    )

    response = openai_client.chat.completions.create(
        model=model_deployment_name,
        messages=[
            {"role": "system", "content": system_text},
            {"role": "user", "content": query}
        ]
    )

    return response.choices[0].message.content


## 5. Try a Query 🎉
Let's do a question about cardio for busy people.


In [35]:
user_query = "What's a good short cardio routine for me if I'm busy?"
answer = rag_chat(user_query)
print("🗣️ User Query:", user_query)
print("🤖 RAG Answer:", answer)

🗣️ User Query: What's a good short cardio routine for me if I'm busy?
🤖 RAG Answer: For cardio endurance, try interval training like HIIT.


## 6. Conclusion
We've demonstrated a **basic RAG** pipeline with:
- **Embedding** docs & storing them in **Azure AI Search**.
- **Retrieving** top docs for user question.
- **Chat** with the retrieved docs.

🔎 You can expand this by adding advanced chunking, more robust retrieval, and quality checks. Enjoy your healthy coding! 🍎